In [ ]:
import h5py

file_name = "lifted_movi_part1_upd1.h5"

file = h5py.File(file_name, "r")

train = file["train"]

train['Subject_11__checking_watch']['PG1'].keys()

<KeysViewHDF5 []>

In [11]:
grp = train["Subject_11__checking_watch"]
# T = 261
grp["PG2"]['trans']

<HDF5 dataset "trans": shape (261, 3), type "<f4">

In [8]:
import scipy.io as sio
import numpy as np

def file_name(file_type, id):
    if file_type == 'mat':
        return f"F_amass_Subject_{id}.mat"
    elif file_type == 'pg1':
        return f"F_PG1_Subject_{id}_L.avi"
    elif file_type == 'pg2':
        return f"F_PG2_Subject_{id}_L.avi"
    elif file_type == 'v3d':
        return f"F_v3d_Subject_{id}.mat"
    else:
        raise Exception("Unrecognized file type")

def _unwrap(x):
    """
    Unwrap nested numpy object arrays from scipy.io.loadmat (squeeze_me=False).
    Only peels single-element (size==1) object arrays, so multi-element arrays
    like move (21,1) are left intact. Stops when we reach a mat_struct,
    a numeric ndarray, or a scalar -- regardless of how many layers deep.
    """
    while isinstance(x, np.ndarray) and x.dtype == object and x.size == 1:
        x = x.flat[0]
    return x


def _scalar(x) -> int | float:
    """Extract a Python scalar from any numpy array shape."""
    if isinstance(x, np.ndarray):
        return x.flat[0].item()
    return x


def _str(x) -> str:
    """Extract a plain Python str from a numpy string scalar or 1-element array."""
    if isinstance(x, np.ndarray):
        return str(x.flat[0])
    return str(x)

def load_v3d_mat(v3d_path):
    try:
        mat = sio.loadmat(str(v3d_path), struct_as_record=False, squeeze_me=False)
    except Exception as e:
        print(f"Cannot load {v3d_path}: {e}")
        return None, None
    top_key = next(k for k in mat if not k.startswith("__"))
    subj    = _unwrap(mat[top_key])   # mat_struct with fields: id, subject, move
    move_arr = _unwrap(subj.move)

    action_names = [str(_unwrap(action[0])).lstrip("['").rstrip("']") for action in move_arr.motions_list]
    action_inds = np.array([[int(tup[0]),int(tup[1])] for tup in move_arr.flags30])

    return action_names, action_inds


fname = "F_Subjects_meta/F_v3d_Subject_16.mat"

actions_names, action_inds = load_v3d_mat(fname)

print(*actions_names,sep="\n")

walking
jumping_jack
crossarms
checking_watch
scratching_head
pointing
running_in_spot
hand_clapping
phone_talking
stretching_rm
vertical_jumping
sideways
sitting_down
jogging
stretching
throw/catch
hand_waving
cross_legged_sitting
kicking
taking_photo
crawling


In [5]:
import os 

pg1_subjects = []
pg2_subjects = []

for fname in os.listdir("videos/PG1_avi"):
    pg1_subjects.append(int(fname.split("F_PG1_Subject_")[1].split("_L.avi")[0]))
for fname in os.listdir("videos/PG2_avi"):
    pg2_subjects.append(int(fname.split("F_PG2_Subject_")[1].split("_L.avi")[0]))

common = list(set(pg1_subjects) & set(pg2_subjects))
pg1_only = list(set(pg1_subjects) - set(pg2_subjects))
pg2_only = list(set(pg2_subjects) - set(pg1_subjects))

print(f"Both: {len(common)}")
print(f"PG1 only: {len(pg1_only)}")
print(f"PG2 only: {len(pg2_only)}")

print(f"PG1 only subjects: {pg1_only}")
print(f"PG2 only subjects: {pg2_only}")

Both: 87
PG1 only: 0
PG2 only: 1
PG1 only subjects: []
PG2 only subjects: [6]
